# OpenHeadMouse: Colab session viewer
Run the next two cells. Import controls appear first, followed by the single head/cursor viewer and six participant summary tables.
Use a CPU runtime; no camera, mouse driver, desktop GUI or GPU is needed.

**Privacy:** recordings uploaded here go to your Google Colab runtime, not to GitHub or any playback service. 
Do not share a notebook with saved outputs containing participant data. Runtime files and imports are temporary; download the summary CSV before disconnecting.

**Setup:** the repository/branch below must contain `colab_session_viewer`. 
If working with unpublished changes, upload the complete project folder to the runtime first and set `PROJECT_DIR` to it.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/simonjudge/OpenHeadMouse.git'
REVISION = ''  # Optional published branch or tag; empty uses the default branch.
# Prefer this notebook's local project over an older nested clone.
PROJECT_DIR = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                    if (folder / 'colab_session_viewer' / 'app.py').is_file()
                    and (folder / 'core').is_dir()), Path.cwd() / 'OpenHeadMouse')

if not PROJECT_DIR.exists():
    command = ['git', 'clone', '--depth', '1']
    if REVISION:
        command += ['--branch', REVISION]
    subprocess.check_call(command + [REPOSITORY_URL, str(PROJECT_DIR)])
requirements = PROJECT_DIR / 'colab_session_viewer' / 'requirements.txt'
if not requirements.is_file():
    raise FileNotFoundError('This checkout does not contain the Colab viewer. Select its published branch or upload the complete changed project.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
PROJECT_DIR = PROJECT_DIR.resolve()
sys.path = [str(PROJECT_DIR)] + [entry for entry in sys.path if entry != str(PROJECT_DIR)]
loaded_viewer = sys.modules.get('colab_session_viewer')
if loaded_viewer is not None and Path(loaded_viewer.__file__).resolve().parent != PROJECT_DIR / 'colab_session_viewer':
    raise RuntimeError('Another viewer checkout is already loaded. Restart the kernel and rerun setup.')

from session_viewer.mesh_visualizer import download_mediapipe_model
download_mediapipe_model()
print('Ready. Run the viewer cell below.')


In [ ]:
try:
    from google.colab import output
except ModuleNotFoundError as error:
    if error.name not in ('google', 'google.colab'):
        raise
    print('Local notebook runtime; Colab widget manager not needed.')
else:
    output.enable_custom_widget_manager()
import importlib
import colab_session_viewer.data as viewer_data
import colab_session_viewer.archive as viewer_archive
import colab_session_viewer.app as viewer_app
for module in (viewer_data, viewer_archive, viewer_app):
    importlib.reload(module)
from colab_session_viewer.app import NotebookViewer
print('Viewer loaded from:', viewer_app.__file__)

viewer = NotebookViewer()
viewer.show();


## Importing multiple participants and sessions
For each test, enter a participant (for example `P01`) and session (`Test 1`) and choose a head JSON/CSV, a Fitts JSON, or both. 
Click **Add session**, then repeat with `P01 / Test 2`, `P02 / Test 1`, etc. Choose both files only when recorded together with overlapping host timestamps. 
Legacy files without host timestamps can be imported separately; they are never silently aligned.

Upload a ZIP **with or without a manifest** using **Dataset ZIP**, then **Import dataset ZIP**. Without a manifest, the viewer detects recordings by their contents and suggests pairs from overlapping shared host timestamps. Review file paths, timestamps, sample/target counts and Fitts notes alongside the editable participant/session labels. The Participant input above the uploads prefills every review row; labels remain editable. Ambiguous matches and legacy streams remain separate; participants are not guessed from timestamps.

Edit the head/Fitts dropdowns to correct pairings, use **Add assignment row** to split pairs, or untick unwanted rows. Each file can be included once per import. Click **Import labelled sessions**, then **Download manifest.csv for ZIP import N** to save the assignments. Put the downloaded manifest at the root of the original ZIP to reuse those labels and pairings. Matching streams stay on their shared host clock, not an artificial start-time alignment.

Any root-level CSV filename (for example `A manifest.csv`) is checked for a valid manifest. A valid manifest imports only its listed files, not other recordings in the ZIP. Multiple valid manifests or malformed assignments produce an explicit error. Unrelated CSVs are not treated as manifests. Required columns:
```csv
participant,session,head_file,fitts_file
P01,Test 1,P01/head1.json,P01/fitts1.json
P01,Test 2,P01/head2.csv,P01/fitts2.json
P02,Test 1,,P02/fitts1.json
```
Paths are relative to the ZIP, use `/` separators, and may be blank when that stream was not recorded. 
You can import multiple ZIPs without clearing earlier data; each has a separate downloadable manifest. Duplicate participant/session names are rejected rather than overwritten. Manifests store assignments, not source data or exclusion settings. Keep the original ZIP as well.

The per-participant exclusion control removes the first **N targets per Fitts file**, not entire tests and not repetitions of each condition. 
Six compact tables each contain one row per participant: **Session summary**, **Initial head positions**, **Range of movement stats**, **Head movement stats**, **Fitts targets and hits**, and **Fitts stats**, with CSV downloads and source-file footnotes. Initial positions show explicit absolute/provider-frame yaw, calibration-heading-relative yaw, and calibration mean yaw. Wireframes use mean calibration yaw with initial-rest pitch/roll. Absolute yaw is not necessarily a compass heading; gravity cannot determine yaw. Pitch/roll retain provider-frame tilt. Without calibration, relative yaw/plots are unavailable but absolute yaw remains. Head movement ranges are quaternion-derived Euler spans, reported as the maximum within-file range per axis, not pooled extrema across different calibrations. Full recordings include preparation/ROM; expand range diagnostics for Fitts-only comparisons and near-gimbal pitch warnings. Euler angles can still be coupled/singular after quaternion conversion. No automatic drift removal is applied. ROM tables remain task-only. Display/export use one decimal place; calculations retain precision. 

**IMU versus camera references:** xIMU3/AirPods initial pitch and roll are gravity-referenced tilt estimates, subject to sensor fusion and mounting/fit; webcam pitch and roll are camera-referenced, not gravity-referenced. xIMU3 yaw may be magnetic when enabled and reliable; AirPods yaw has an arbitrary heading reference. Expand **Angle references and zero-position diagrams** in the initial-position section for head-mesh diagrams of pitch/roll zero, absolute/calibration yaw zero (provider heading), and relative yaw zero (mean calibration heading). Its example calibration heading is illustrative, not participant data. Movement ranges and playback use neutral-relative Euler components, not gravity-referenced tilt ranges; a tilted neutral can couple heading changes into pitch/roll.

Participant pose plots include signed head axes: X forward, Y left, Z up. Positive yaw turns left, positive pitch looks up, positive roll tilts the left ear down; negative values reverse those directions. **Zero yaw / Zero pitch / Zero roll** controls above the summaries redraw every participant plot with selected angles set to 0.0 degrees; uncheck to restore. These overrides affect plots only, never tables, CSVs or playback. Source-file lists, angle references, calculation notes and the **Field key** under every table start collapsed. Expand each key for field definitions, units, aggregation and Fitts/path formulas. Table columns fit their content and headers are bold.

Playback, head measurements and pre-test summaries retain all data. If N exhausts a file, that file contributes zero pointing targets; unavailable measures are blank.

The viewer has play/pause, rewind, seek, 0.1–5x speed, graph zoom, a rotatable head mesh, live values, target/click markers and pre-test playback. 
The browser renderer replaces Qt/OpenGL; recording times, neutral pose, coordinate transforms, angular dynamics and summary formulas follow the desktop viewer.

For recordings already on Google Drive, mount Drive in a separate cell and call:
```python
viewer.add_session('P03', 'Test 1',
    head_path='/content/drive/MyDrive/study/head.json',
    fitts_path='/content/drive/MyDrive/study/fitts.json')
```
Only run the viewer cell once per collection: rerunning it starts a new empty collection.